# Padding and Stride

Recall the example of a convolution in the figure. 
The input had both a height and width of 3
and the convolution kernel had both a height and width of 2,
yielding an output representation with dimension $2\times2$.
Assuming that the input shape is $n_\textrm{h}\times n_\textrm{w}$
and the convolution kernel shape is $k_\textrm{h}\times k_\textrm{w}$,
the output shape will be $(n_\textrm{h}-k_\textrm{h}+1) \times (n_\textrm{w}-k_\textrm{w}+1)$: 
we can only shift the convolution kernel so far until it runs out
of pixels to apply the convolution to. 

In the following we will explore a number of techniques,
including padding, strided convolutions, and dilation,
that offer more control over the size of the output
and over the area of the input that each output element sees. 
As motivation, note that since kernels generally
have width and height greater than $1$,
after applying many successive convolutions,
we tend to wind up with outputs that are
considerably smaller than our input.
If we start with a $240 \times 240$ pixel image,
ten unpadded layers of $5 \times 5$ convolutions
reduce the representation to $200 \times 200$ pixels. The surviving outputs
are centered away from the original boundary, and boundary pixels influence
far fewer activations than interior pixels.
*Padding* is the most popular tool for handling this issue.
In other cases, we may want to reduce the dimensionality drastically,
e.g., if we find the original input resolution to be unwieldy.
*Strided convolutions* are a popular technique that can help in these instances.

In [1]:
import torch
from torch import nn

## Padding

As described above, one tricky issue when applying convolutional layers
is that we tend to lose pixels on the perimeter of our image. Consider the figure that depicts the pixel utilization as a function of the convolution kernel size and the position within the image. The pixels in the corners are hardly used at all. 

![Pixel utilization for convolutions of size $1 \times 1$, $2 \times 2$, and $3 \times 3$ respectively.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-reuse.svg)

Since we typically use small kernels,
for any given convolution
we might only lose a few pixels
but this can add up as we apply
many successive convolutional layers.
One straightforward solution to this problem
is to add extra pixels of filler around the boundary of our input image,
thus increasing the effective size of the image.
Typically, we set the values of the extra pixels to zero.
In the figure, we pad a $3 \times 3$ input,
increasing its size to $5 \times 5$.
The corresponding output then increases to a $4 \times 4$ matrix.
The shaded portions are the first output element as well as the input and kernel tensor elements used for the output computation: $0\times0+0\times1+0\times2+0\times3=0$.

![Two-dimensional cross-correlation with padding.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-pad.svg)

In general, if we add a total of $p_\textrm{h}$ rows of padding
(roughly half on top and half on bottom)
and a total of $p_\textrm{w}$ columns of padding
(roughly half on the left and half on the right),
the output shape will be

$$(n_\textrm{h}-k_\textrm{h}+p_\textrm{h}+1)\times(n_\textrm{w}-k_\textrm{w}+p_\textrm{w}+1).$$

This means that the height and width of the output
will increase by $p_\textrm{h}$ and $p_\textrm{w}$, respectively.

In many cases, we will want to set $p_\textrm{h}=k_\textrm{h}-1$ and $p_\textrm{w}=k_\textrm{w}-1$
to give the input and output the same height and width.
This will make it easier to predict the output shape of each layer
when constructing the network.
Assuming that $k_\textrm{h}$ is odd here,
we will pad $p_\textrm{h}/2$ rows on both sides of the height.
If $k_\textrm{h}$ is even, one possibility is to
pad $\lceil p_\textrm{h}/2\rceil$ rows on the top of the input
and $\lfloor p_\textrm{h}/2\rfloor$ rows on the bottom.
We will pad both sides of the width in the same way.

CNNs commonly use convolution kernels
with odd height and width values, such as 1, 3, 5, or 7.
Choosing odd kernel sizes has the benefit
that we can preserve the dimensionality
while padding with the same number of rows on top and bottom,
and the same number of columns on left and right.

Moreover, this practice of using odd kernels
and padding to precisely preserve dimensionality
offers a clerical benefit.
For any two-dimensional tensor `X`,
when the kernel's size is odd
and the number of padding rows and columns
on all sides are the same,
thereby producing an output with the same height and width as the input,
we know that the output `Y[i, j]` is calculated
by cross-correlation of the input and convolution kernel
with the window centered on `X[i, j]`.

In the following example, we create a two-dimensional convolutional layer
with a height and width of 3
and apply 1 pixel of padding on all sides.
Given an input with a height and width of 8,
we find that the height and width of the output is also 8.

In [2]:
# We define a helper function to calculate convolutions. It initializes the
# convolutional layer weights and performs corresponding dimensionality
# elevations and reductions on the input and output
def comp_conv2d(conv2d, X):
    # (1, 1) indicates that batch size and the number of channels are both 1
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    # Strip the first two dimensions: examples and channels
    return Y.reshape(Y.shape[2:])

# 1 row and column is padded on either side, so a total of 2 rows or columns
# are added
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1)
X = torch.rand(size=(8, 8))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

When the height and width of the convolution kernel are different,
we can make the output and input have the same height and width
by setting different padding numbers for height and width.

In [3]:
# We use a convolution kernel with height 5 and width 3. The padding on either
# side of the height and width are 2 and 1, respectively
conv2d = nn.LazyConv2d(1, kernel_size=(5, 3), padding=(2, 1))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

## Stride

When computing the cross-correlation,
we start with the convolution window
at the upper-left corner of the input tensor,
and then slide it over all locations both down and to the right.
In the previous examples, we defaulted to sliding one element at a time.
However, sometimes, either for computational efficiency
or because we wish to downsample,
we move our window more than one element at a time,
skipping the intermediate locations. This is particularly useful if the convolution 
kernel is large since it captures a large area of the underlying image.

We refer to the number of rows and columns traversed per slide as *stride*.
So far, we have used strides of 1, both for height and width.
Sometimes, we may want to use a larger stride.
the figure shows a two-dimensional cross-correlation operation
with a stride of 3 vertically and 2 horizontally.
The shaded portions are the output elements as well as the input and kernel tensor elements used for the output computation: $0\times0+0\times1+1\times2+2\times3=8$, $0\times0+6\times1+0\times2+0\times3=6$.
We can see that when the second element of the first column is generated,
the convolution window slides down three rows.
The convolution window slides two columns to the right
when the second element of the first row is generated.
When the convolution window continues to slide two columns to the right on the input,
there is no output because the input element cannot fill the window
(unless we add another column of padding).

![Cross-correlation with strides of 3 and 2 for height and width, respectively.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-stride.svg)

In general, when the stride for the height is $s_\textrm{h}$
and the stride for the width is $s_\textrm{w}$, the output shape is

$$\lfloor(n_\textrm{h}-k_\textrm{h}+p_\textrm{h}+s_\textrm{h})/s_\textrm{h}\rfloor \times \lfloor(n_\textrm{w}-k_\textrm{w}+p_\textrm{w}+s_\textrm{w})/s_\textrm{w}\rfloor.$$

If we set $p_\textrm{h}=k_\textrm{h}-1$ and $p_\textrm{w}=k_\textrm{w}-1$,
then the output shape can be simplified to
$\lfloor(n_\textrm{h}+s_\textrm{h}-1)/s_\textrm{h}\rfloor \times \lfloor(n_\textrm{w}+s_\textrm{w}-1)/s_\textrm{w}\rfloor$.
Going a step further, if the input height and width
are divisible by the strides on the height and width,
then the output shape will be $(n_\textrm{h}/s_\textrm{h}) \times (n_\textrm{w}/s_\textrm{w})$.

Below, we set the strides on both the height and width to 2,
thus halving the input height and width.

In [4]:
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1, stride=2)
comp_conv2d(conv2d, X).shape

torch.Size([4, 4])

Let's look at a slightly more complicated example.

In [5]:
conv2d = nn.LazyConv2d(1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
comp_conv2d(conv2d, X).shape

torch.Size([2, 2])

## Dilation

Padding and stride change where the kernel is applied. A third modification
changes the kernel's footprint itself: a *dilated convolution*
[@yu2016dilated] places the kernel taps $d$ pixels apart
instead of at adjacent positions,
where $d$ is called the *dilation* (or *dilation rate*).
A $k_\textrm{h} \times k_\textrm{w}$ kernel with dilation $d$
still has $k_\textrm{h} k_\textrm{w}$ weights,
but along each axis the taps now span a window of

$$k' = k + (k-1)(d-1) = d(k-1)+1$$

pixels, the *effective kernel size*.
Setting $d=1$ recovers the ordinary convolution.
the figure shows a $3 \times 3$ kernel at dilations 1 and 2:
at dilation 2 the nine taps spread over a $5 \times 5$ footprint,
skipping every other pixel,
while the cost of applying the kernel is unchanged.

![A 3×3 kernel at dilation 1 and dilation 2: the nine taps spread over a 5×5 footprint while the cost stays fixed.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-conv-dilation.svg)

For output shapes, a dilated kernel behaves exactly like a dense kernel
of its effective size. Substituting $k'$ into the strided formula above
gives the general form covering padding, stride, and dilation at once:

$$\lfloor(n_\textrm{h}-d_\textrm{h}(k_\textrm{h}-1)+p_\textrm{h}+s_\textrm{h}-1)/s_\textrm{h}\rfloor \times \lfloor(n_\textrm{w}-d_\textrm{w}(k_\textrm{w}-1)+p_\textrm{w}+s_\textrm{w}-1)/s_\textrm{w}\rfloor.$$

With $d_\textrm{h}=d_\textrm{w}=1$ this reduces to the strided formula,
and with unit stride and no padding to the $(n-k+1)$ rule we started from.

The point of dilation is receptive-field growth on a budget.
In the receptive-field formula the equation,
each layer contributes $k'_i - 1 = d_i(k_i - 1)$ at stride 1,
so doubling the dilation at every layer,
$d_i = 1, 2, 4, \ldots, 2^{L-1}$ for $3 \times 3$ kernels,
yields a receptive field of side $2^{L+1}-1$:
it grows exponentially with depth
while every layer costs the same nine multiplications per output.
The alternative route to a large receptive field, striding,
pays for it by shrinking the output.
This makes dilation the standard tool for *dense prediction* tasks
such as semantic segmentation,
where every pixel needs wide context
but the output must keep the input's resolution;
we will meet it again in the fully convolutional networks
of that section.

Let's verify the effective-size claim numerically:
on our $8 \times 8$ input,
a $3 \times 3$ kernel at dilation 2
produces the same $4 \times 4$ output
as a dense $5 \times 5$ kernel.

In [6]:
# Dilation 2 gives the 3x3 kernel a 5x5 footprint, so both convolutions
# shrink the 8x8 input to 4x4
conv2d = nn.LazyConv2d(1, kernel_size=3, dilation=2)
conv5 = nn.LazyConv2d(1, kernel_size=5)
comp_conv2d(conv2d, X).shape, comp_conv2d(conv5, X).shape

(torch.Size([4, 4]), torch.Size([4, 4]))

## Summary and Discussion

Padding can increase the height and width of the output and is often chosen so
that input and output have the same spatial shape. It lets boundary pixels
affect outputs centered near the boundary, but does not make their use identical
to that of interior pixels: a corner pixel still belongs to fewer windows
containing real image values. Typically we pick symmetric padding on both sides
of the input height and width. In this case we refer to
$(p_\textrm{h}, p_\textrm{w})$ padding. Most commonly we set
$p_\textrm{h} = p_\textrm{w}$, in which case we simply state that we choose
padding $p$.

A similar convention applies to strides. When the vertical stride $s_\textrm{h}$ and horizontal stride $s_\textrm{w}$ match, we simply talk about stride $s$. The stride can reduce the resolution of the output, for example reducing the height and width of the output to only $1/n$ of the height and width of the input for $n > 1$. The same shorthand covers dilation: when $d_\textrm{h} = d_\textrm{w}$ we speak of dilation $d$. Dilation widens the window each output sees without adding weights or reducing resolution, which is what dense prediction needs. By default, the padding is 0 and the stride and dilation are 1.

So far all padding that we discussed simply extended images with zeros. This is cheap, and operators can be engineered to handle it implicitly without allocating a larger tensor, but it is not a neutral choice: zero padding is a *boundary condition*, an assertion that the world outside the image is black. Filters whose window overlaps the border therefore see systematically different inputs from filters in the interior, and the resulting artifacts propagate deep into the feature maps [@Alsallakh.Kokhlikyan.Miglani.ea.2020]. Padding also leaks absolute position: convolution itself cannot tell where in the image it is, but a unit near the border can infer its location from how much zero "whitespace" it sees, and trained CNNs do exploit this signal. Alternatives such as reflect padding (mirroring the border rows and columns) or replicate padding (repeating them) remove the artificial black frame; in practice they are reached for when border artifacts become visible, for example in image generation.


## Exercises

1. Given the final code example in this section with kernel size $(3, 5)$, padding $(0, 1)$, and stride $(3, 4)$, 
   calculate the output shape to check if it is consistent with the experimental result.
1. For audio signals, what does a stride of 2 correspond to?
1. Implement mirror padding, i.e., padding where the border values are simply mirrored to extend tensors. 
1. What are the computational benefits of a stride larger than 1?
1. What might be statistical benefits of a stride larger than 1?
1. How would you implement a stride of $\frac{1}{2}$? What does it correspond to? When would this be useful? Compare your answer with the transposed convolutions of that section.
1. A network stacks four $3 \times 3$ convolutions with stride 1 and dilations $1, 2, 4, 8$. Use the equation with each kernel replaced by its effective size to compute the receptive field of one output element. Which pixels inside that field does the output actually depend on? When does this *gridding* effect become a problem, and how would you choose a dilation schedule that avoids it?

[Discussions](https://d2l.discourse.group/t/68)